<a href="https://colab.research.google.com/github/veerasurajj/air-pollution-analysis/blob/main/train_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import joblib

FILENAME = "AirQualityUCI.csv"

df = pd.read_csv(FILENAME, sep=",")

df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print("Raw shape:", df.shape)
print(df.head())

df = df.replace(-200, np.nan)

df = df.dropna(subset=["CO(GT)", "NO2(GT)"])

feature_cols = [
    "CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)", "PT08.S2(NMHC)",
    "NOx(GT)", "PT08.S3(NOx)", "NO2(GT)", "PT08.S4(NO2)", "PT08.S5(O3)",
    "T", "RH", "AH"
]
feature_cols = [c for c in feature_cols if c in df.columns]

for col in feature_cols:
    df[col] = df[col].fillna(df[col].median())

def co_category(co):
    if co < 1:
        return 0
    elif co < 2:
        return 1
    elif co < 10:
        return 2
    else:
        return 3

def no2_category(no2):
    if no2 < 40:
        return 0
    elif no2 < 80:
        return 1
    elif no2 < 180:
        return 2
    else:
        return 3

labels = ["Good", "Moderate", "Poor", "Severe"]

df["aqi_category"] = df.apply(
    lambda row: labels[max(co_category(row["CO(GT)"]), no2_category(row["NO2(GT)"]))],
    axis=1
)

print("\nClass distribution:")
print(df["aqi_category"].value_counts())

X = df[feature_cols]
y = df["aqi_category"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

model = RandomForestClassifier(
    n_estimators=300, max_depth=None, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

importances = pd.Series(model.feature_importances_, index=feature_cols)
print("\nFeature importances:")
print(importances.sort_values(ascending=False))

joblib.dump(model, "aqi_model.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")

print("\nSaved: aqi_model.pkl, label_encoder.pkl, feature_cols.pkl")

Raw shape: (9471, 15)
        Date      Time  CO(GT)  PT08.S1(CO)  NMHC(GT)  C6H6(GT)  \
0  3/10/2004  18:00:00     2.6       1360.0     150.0      11.9   
1  3/10/2004  19:00:00     2.0       1292.0     112.0       9.4   
2  3/10/2004  20:00:00     2.2       1402.0      88.0       9.0   
3  3/10/2004  21:00:00     2.2       1376.0      80.0       9.2   
4  3/10/2004  22:00:00     1.6       1272.0      51.0       6.5   

   PT08.S2(NMHC)  NOx(GT)  PT08.S3(NOx)  NO2(GT)  PT08.S4(NO2)  PT08.S5(O3)  \
0         1046.0    166.0        1056.0    113.0        1692.0       1268.0   
1          955.0    103.0        1174.0     92.0        1559.0        972.0   
2          939.0    131.0        1140.0    114.0        1555.0       1074.0   
3          948.0    172.0        1092.0    122.0        1584.0       1203.0   
4          836.0    131.0        1205.0    116.0        1490.0       1110.0   

      T    RH      AH  
0  13.6  48.9  0.7578  
1  13.3  47.7  0.7255  
2  11.9  54.0  0.7502  
3  1